[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alex-robolabs/override-referee/blob/main/Override_Referee_Lab.ipynb)

# 🦓 Override Referee: Build the AI That Actually Read the Rules
**Robolabs Summer Academy 2026 · Day 4 · Agents: Tools, Knowledge & Loops**

Friday-scrimmage scenario: your alliance partner swears yellow pins are 5 points. You swear they are 10. The manual is 139 pages and the match starts in three minutes. Today you build the referee that settles it.

| Act | Minutes | What happens |
|---|---|---|
| 0 | 2 | Ask a model about Override. Watch it make things up. |
| 1 | 4 | Keyword search: the 2005 fix. Break it. |
| 2 | 6 | Semantic search: Tuesday's meaning map, working for you |
| 3 | 8 | The retriever becomes a **tool** in your Monday agent loop |
| 4 | 5 | The **eval**: stop arguing, start measuring |

**Before you start:**
1. Click **File → Save a copy in Drive** so you have your own editable copy.
2. Work in pairs: one person drives, one navigates. Swap after Act 2.
3. Run cells top to bottom with the ▶ button. You cannot break anything.
4. ⭐ challenges are for everyone. ⭐⭐ and ⭐⭐⭐ are for when you want more.

*Game Manual text © 2026 VEX Robotics, Inc. Used with permission for educational purposes at Robolabs Summer Academy.*

## The one fact this whole lab stands on

The Override Game Manual is **Version 1.0, released July 2, 2026**. Every AI model you can talk to today finished training before that date. **No model on Earth has read this manual.** Whatever it says about Override, it is reconstructing from older VEX games and vibes.

Nothing you learned Tuesday about hallucination should make this surprising. Everything in this lab exists to fix it.

In [ ]:
# @title 🔑 Setup: run me first! { display-mode: "form" }
LANE = "class"  # @param ["class", "own"]
# "class" = shared class key (Gemini, from the projector)
# "own"   = your personal Hugging Face token (see Monday's take-home section)

%pip install -q -U openai sentence-transformers

import json, urllib.request
from openai import OpenAI
from getpass import getpass

# The rulebook, pre-chunked: one entry per rule, extracted from the real
# Override Game Manual v1.0. Parsing the PDF happened at build time, not here.
RULES_URL = "https://raw.githubusercontent.com/alex-robolabs/override-referee/main/rules.json"
ALL_CHUNKS = json.load(urllib.request.urlopen(RULES_URL))
CHUNKS = [c for c in ALL_CHUNKS if c["program"] == "V5RC"]   # VEX U stays out (for now)
families = sorted({c["family"] for c in CHUNKS})
print(f"📖 Loaded the Override manual: {len(CHUNKS)} chunks ({len(ALL_CHUNKS)} with VEX U), families: {', '.join(families)}")

if LANE == "class":
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
    MODEL = "gemini-flash-latest"  # instructors: verify current model name before class
    prompt_text = "Paste the CLASS API key from the projector, or press Enter for replay mode: "
else:
    BASE_URL = "https://router.huggingface.co/v1"
    MODEL = "Qwen/Qwen2.5-7B-Instruct"  # open model; instructors: verify tool-call support
    prompt_text = "Paste YOUR Hugging Face token (starts with hf_), or press Enter for replay mode: "

try:
    _key = getpass(prompt_text).strip()
except Exception:      # no prompt available (e.g. automated runs): replay mode
    _key = ""
REPLAY = (_key == "")
client = OpenAI(base_url=BASE_URL, api_key=_key or "replay")

if not REPLAY:
    try:
        ping = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "Reply with exactly: Ref crew, ready! 🦓"}],
        )
        print(f"Lane: {LANE} · Model: {MODEL}")
        print(ping.choices[0].message.content)
    except Exception as e:
        REPLAY = True
        print("API unreachable, no problem. Error:", e)

if REPLAY:
    print("🎬 REPLAY MODE: Acts 0 and 3 will print runs recorded earlier with the real model.")
    print("   Acts 1, 2 and 4 (search + eval) always run live on your machine. No key needed.")

---
# Act 0 · The model has never read this manual 🙈

One question. No tools, no manual, just the model. Then we put its answer next to what the rulebook actually says.

In [ ]:
# @title 🤖 Ask the naked model an Override question
QUESTION = "In the VEX V5 Robotics Competition 2026-2027 game Override, how many points is a Scored yellow Pin worth?"

REPLAY_ACT0 = """[PLACEHOLDER PENDING LIVE CAPTURE: do not ship]"""

if REPLAY:
    answer = REPLAY_ACT0
    print("🎬 REPLAY (recorded earlier from the real model):\n")
else:
    answer = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": QUESTION}],
    ).choices[0].message.content

print("❓", QUESTION)
print()
print("🤖 THE MODEL SAYS (confidently):")
print(answer)
print()
truth = next(c for c in CHUNKS if c["id"] == "Match Scoring Values")
print(f"📖 THE MANUAL SAYS ({truth['id']}, manual p. {truth['page']}):")
print(truth["text"])

The model answered like it knew. It did not know. **This manual came out after its training ended. It has never seen a single word of it.**

Your competition robot will be judged by these exact rules all season. An AI that guesses about them is worse than no AI. The fix is not a smarter model. The fix is giving the model the book, and that is the rest of this lab.

---
# Act 1 · Keyword search: the 2005 fix 🔎

Before AI could read meaning, search meant counting shared words. It is fast, free, and it works great, right up until it doesn't. Run it.

In [ ]:
# @title 🔎 keyword_search: score every rule by shared words
import re

STOP = {"a", "an", "the", "is", "are", "in", "on", "of", "to", "and", "or",
        "for", "with", "at", "by", "be", "it", "its", "my", "our", "we", "i",
        "you", "your", "do", "does", "can", "what", "how", "that", "this"}

def chunk_text(c):
    return f"{c['id']}: {c['title']} {c['text']}"

def keyword_score(question, c):
    q_words = set(re.findall(r"[a-z0-9]+", question.lower())) - STOP
    c_words = set(re.findall(r"[a-z0-9]+", chunk_text(c).lower())) - STOP
    return len(q_words & c_words)

def keyword_search(question, k=3):
    ranked = sorted(CHUNKS, key=lambda c: -keyword_score(question, c))
    return [(keyword_score(question, c), c) for c in ranked[:k]]

def show(results):
    for score, c in results:
        print(f"   {score:6.3f}  [{c['id']}] {c['title'][:70]}")

# Questions that use the manual's own vocabulary. Keyword search crushes these:
for q in ["How many points is a yellow Pin worth?",
          "Is horizontal expansion limited?",
          "What is the maximum number of Pins a Robot may have Possession of?"]:
    print("❓", q)
    show(keyword_search(q))
    print()

In [ ]:
# @title 💥 Now break it
# Ask something every VEX kid asks, in words the manual never uses:
my_question = "How tall can my robot get during a match?"

print("❓", my_question)
show(keyword_search(my_question))
print()
print("The governing rule is <SG3> Vertical expansion is limited. Did you see it up there?")

Keyword search just whiffed. You said **tall**. The manual says **vertical expansion** and **overall height of 50 inches**. Zero shared words that matter, so the right rule cannot win, no matter how obviously your question is about it. Same disease as Monday's `RULES` list: exact wording, hand-predicted.

**⭐ Challenge 1:** Edit `my_question` and find two more questions a referee would understand instantly but keyword search fumbles. (Hint: ask about *pulling pins out of the other team's goal*, or *whether code needs the competition template*.)

---
# Act 2 · Semantic search: the meaning map 🗺️

Tuesday you saw it: **meaning is a place on a map.** Every sentence becomes a point; nearby points mean nearby things. "How tall can my robot get" and "vertical expansion is limited" live in the same neighborhood of that map even though they share no words.

Now we embed all ~200 rule chunks once, and answer questions by distance.

In [ ]:
# @title 🗺️ Embed the whole rulebook (runs once, no key needed)
from sentence_transformers import SentenceTransformer
import numpy as np

print("🧠 Loading the embedding model (all-MiniLM-L6-v2, ~90 MB, one time)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print(f"🗺️ Placing {len(CHUNKS)} rules on the meaning map...")
EMBEDDINGS = embedder.encode([chunk_text(c) for c in CHUNKS],
                             normalize_embeddings=True, show_progress_bar=False)
print(f"Done. Every rule is now a point in {EMBEDDINGS.shape[1]}-dimensional space.")

def semantic_search(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    sims = EMBEDDINGS @ q          # cosine similarity to every rule at once
    top = np.argsort(-sims)[:k]
    return [(float(sims[i]), CHUNKS[i]) for i in top]

In [ ]:
# @title 🎯 The exact question that broke Act 1
print("❓", my_question)
print()
print("KEYWORD (Act 1):")
show(keyword_search(my_question))
print()
print("SEMANTIC (Act 2):")
show(semantic_search(my_question))
print()
top = semantic_search(my_question, k=1)[0][1]
print(f"📖 [{top['id']}] {top['title']} (manual p. {top['page']})")
print(top["text"][:400])

Same rulebook. Same question. The map found **<SG3> Vertical expansion is limited** because *tall robot* and *vertical height limit* mean the same thing, and meaning is what got embedded.

**⭐⭐ Challenge 2:** Semantic search is not magic. Find a question it still gets wrong (the top 3 miss the rule you needed). Real referee questions that fail exist in this manual. Two hints from our own testing: short definition chunks sometimes outshout the real rule, and the word *match* pulls everything toward the Match definition. When you find one, write one sentence on WHY it failed.

---
# Act 3 · The retriever becomes a tool ⚙️

Monday's agent checked batteries with `check_battery`. Same loop, new tool: `search_rules`. The agent decides **when** to search, **what** to search for, and **whether one search is enough.** That last one matters: a real referee question often needs a legality rule AND a scoring value, which live in different chunks.

In [ ]:
# @title 🧰 Wrap the retriever as a tool
def search_rules(query, k=3):
    """Search the Override Game Manual. Returns the top rules for a query."""
    out = []
    for score, c in semantic_search(query, k):
        out.append(f"[{c['id']}] {c['title']} (manual p. {c['page']})\n{c['text'][:400]}")
    return "\n\n".join(out)

tool_declarations = [
    {"type": "function", "function": {
        "name": "search_rules",
        "description": "Search the official Override Game Manual for rules, scoring values, and definitions. Ask one focused question per call.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string", "description": "What to look up, e.g. 'points for robot in midfield'"}},
                       "required": ["query"]}}},
]
TOOLS = {"search_rules": search_rules}

REFEREE_PERSONA = (
    "You are a referee assistant for Override, the VEX V5 Robotics Competition "
    "2026-2027 game. You have NOT memorized this manual: it is newer than your "
    "training. ALWAYS use the search_rules tool before answering. Every claim "
    "must cite a rule ID like <SG3> and briefly quote the governing words. "
    "Questions about legality AND points need one search per part. If the "
    "retrieved rules do not clearly answer, say 'not covered by the manual' "
    "and recommend asking the Head Referee. Keep answers under 120 words."
)

print("Tool ready. Try it directly, no agent yet:")
print(search_rules("how tall can a robot expand", k=1))

In [ ]:
# @title 🔄 The referee agent (Monday's loop, Thursday's tool)
REPLAY_ACT3 = """[PLACEHOLDER PENDING LIVE CAPTURE: do not ship]"""

def run_referee(question, max_steps=5):
    print(f"❓ QUESTION: {question}")
    print("─" * 60)
    messages = [
        {"role": "system", "content": REFEREE_PERSONA},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tool_declarations,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:  # no more searches needed: the ref is ready to rule
            print("─" * 60)
            print("✅ FINAL ANSWER:")
            print(msg.content)
            return

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name,
                                         "arguments": tc.function.arguments}}
                           for tc in msg.tool_calls],
        })
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"🧠 Step {step} · PLAN: the agent decided to call {tc.function.name}({args})")
            result = TOOLS[tc.function.name](**args)
            preview = result[:180].replace("\n", " ")
            print(f"🔧            ACT: tool returned → {preview}...")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        print("👀            OBSERVE: sending results back to the model...")
        print()
    print("⚠️ Stopped after max steps.")

# The star question. Watch the step count: it should search TWICE,
# once for the midfield rule, once for the points.
STAR = "Our robot was leaning over the midfield at the buzzer. Does that count as in the midfield, and how many points is it worth?"

if REPLAY:
    print("🎬 REPLAY (recorded earlier from the real model):\n")
    print(REPLAY_ACT3)
else:
    run_referee(STAR)

Count the PLAN lines. The agent split the question itself: one search for *does leaning over the midfield count* (it found **<SC6>**, the infinite 3D projection rule), a second search for *how many points* (it found the scoring table: **8 points**). Nobody wrote an if-statement for that. The loop decided.

**⭐ Challenge 3:** Ask the referee your own question. Steal one from Challenge 1, or try "Can we shoot pins across the field into a goal?"

**⭐⭐ Challenge 4:** Right now the persona *asks nicely* for citations. Make it refuse: edit `REFEREE_PERSONA` so that if a draft answer has no rule ID in it, the agent must keep searching or answer exactly "not covered by the manual". Test it on something weird ("can our robot carry a walkie talkie?").

---
# Act 4 · The eval: stop arguing, start measuring 📊

Is semantic search actually better, or did we cherry-pick three demos? That question has a professional answer: **an eval.** Eight real referee questions, each with the rule that governs it. A search passes a question if the right rule is in its top 3. No opinions. Just a score.

In [ ]:
# @title 📊 Eight questions, two search engines, one table
EVAL_SET = [
    ("How tall can my robot get during a match?", ["SG3"]),
    ("Can my robot hold two pins at the same time?", ["SG6"]),
    ("Our robot was leaning over the midfield at the buzzer. Does that count as in the midfield, and how many points is it worth?", ["SC6", "Match Scoring Values", "Midfield"]),
    ("Can I pull scoring objects out of the other alliance's goal?", ["SG9"]),
    ("Does my robot's code have to use the competition template?", ["R9"]),
    ("How many points is the autonomous bonus worth?", ["Match Scoring Values", "SC7", "Autonomous Bonus"]),
    ("Our robot tipped over, can a person reach into the field to fix it?", ["GG4"]),
    ("Who has the final say if we disagree with a call?", ["T1", "T3"]),
]

def run_eval(search_fn, k=3):
    hits = 0
    marks = []
    for question, expected in EVAL_SET:
        top_ids = [c["id"] for _, c in search_fn(question, k)]
        ok = any(e in top_ids for e in expected)
        hits += ok
        marks.append("✅" if ok else "❌")
    return hits, marks

kw_hits, kw_marks = run_eval(keyword_search)
sem_hits, sem_marks = run_eval(semantic_search)

print(f"{'question':62} {'keyword':8} semantic")
print("─" * 82)
for (q, _), km, sm in zip(EVAL_SET, kw_marks, sem_marks):
    print(f"{q[:60]:62} {km:7} {sm}")
print("─" * 82)
print(f"{'HIT AT TOP 3':62} {kw_hits}/8      {sem_hits}/8")

**This is an eval: a unit test for behavior.** You did not argue about which search is better. You measured it. Every serious AI team lives and dies by tables like this one, and Friday's judges are an eval too: a rubric, applied to your project, producing a score.

**⭐⭐⭐ Challenge 5: eval-driven development.** Make the eval better, then make the system beat it.
1. Add this question to `EVAL_SET`: `("Are points counted the moment the match ends?", ["SC1"])` and re-run. Semantic search misses it (we checked).
2. Now fix the SYSTEM without touching the model. The problem is the chunk: SC1's title is referee-speak. Give it words humans say. Run this, then re-run the Act 2 embedding cell, then the eval cell:
```python
for c in CHUNKS:
    if c["id"] == "SC1":
        c["title"] = "When points are counted: scores are evaluated the moment the Match ends, after everything settles"
```
3. Watch the score move. You just did eval-driven development: the loop every AI product team runs all day, every day.

---
# 🎁 The gift: this system, in your pocket, all season

Everything you just built (the chunked manual, the embeddings, the meaning map search) is live at:

## **https://alex-robolabs.github.io/override-referee/**

![QR code to Override Referee](https://raw.githubusercontent.com/alex-robolabs/override-referee/main/qr-code.png)

Same rules.json. Same embedding model, running **in your phone's browser**, no key, no server, free forever. It is yours for the season: pit crew disputes, inspection prep, "wait is that legal" moments at 8am on a Saturday. The rule cards cite the manual page so you can prove it to the ref.

**Take it home tonight:** you already own the whole stack from Monday (your Hugging Face token). This notebook runs on it: flip `LANE` to `"own"` in Setup. And the web app's source is one HTML file, [readable here](https://github.com/alex-robolabs/override-referee): view source, it is the same `semantic_search` you wrote in Act 2.

---
### 📋 Facilitator notes (Iniya)

**Timing (25 min):** Setup 3 · Act 0 two · Act 1 four · Act 2 five · Act 3 seven · Act 4 four. The wrap and gift reveal ride on the closing slide. If running long: Act 1's warmup cell is skippable (go straight to break-it), and Challenge cells are never run live.

**The two-line launch pitch:** "Monday your agent could check a battery. Today it reads the 139-page rulebook that no AI on Earth has ever seen, and by the end you will have measured, not argued, that it works."

**Setup beats (do these before students arrive):** run the notebook once yourself with the class key (model downloads cache per runtime, not per account, so each student still downloads ~90 MB: that is fine, it is fast on Colab). Confirm `gemini-flash-latest` still resolves. Class key on the projector, same flow as Monday. Delete the key Friday.

**Replay mode is a feature:** no key or no wifi means Acts 0 and 3 print recorded runs and the lab still lands every beat. Acts 1, 2, 4 never need a key. If the API 429s mid-class, have tables run one at a time for a minute, or just switch those two cells to replay (re-run Setup, press Enter at the key prompt).

**Checkpoints to call from the front:** "everyone seen the model lie about yellow pins?" → "everyone broken keyword search?" → "everyone seen SG3 come back?" → "everyone watched it search twice?" → "everyone's table shows 3 vs 8?"

**Expected numbers:** keyword 3/8, semantic 8/8 on the eval. Yellow pin = 10 points, autonomous bonus = 12, midfield robot = 8 (scoring table, manual p. 25). The break-it question's governing rule is <SG3> (50 inch limit, manual p. 31).

**Challenge answer key:**
- **C1 (fool keyword):** verified examples: "Can I pull scoring objects out of the other alliance's goal?" (needs <SG9>, keyword misses), "Does my robot's code have to use the competition template?" (needs <R9>, keyword misses), "Our robot tipped over, can a person reach into the field to fix it?" is NOT a valid answer (keyword finds <GG4>: "field" and "match" overlap).
- **C2 (fool semantic):** verified examples: "Is it legal to hold an opposing robot in place all match?" (top hit is the Head-to-Head Match definition: the word *match* dominates, and short definition chunks sit close to short questions). "How long is the autonomous period?" (the Autonomous Period definition outranks the Match Timing chunk that contains the actual number). Why: embeddings average the whole sentence; the strongest noun wins, and definitions are pure concentrated nouns.
- **C4 (hard citations):** add to the persona: "If your draft answer contains no rule ID in angle brackets, do not send it: search again with different words, and if two searches still do not answer, reply exactly 'not covered by the manual'." Walkie talkie test should end at <R18> Prohibited Items or 'not covered by the manual', either is a win.
- **C5 (eval-driven development):** the SC1 title fix is verified: semantic goes 8/9 → 9/9 after re-embedding. If a student asks why we re-run the embedding cell: the map does not move until you re-place the point.

**Common snags:** key pasted with a space (re-run Setup); student edited a search function and got NameError (Runtime → Run all); `sentence-transformers` install takes ~40s on a fresh runtime, narrate over it; a pair stuck on challenges (optional, say so).

**VEX U easter egg (fast finishers):** `CHUNKS` filtered out 39 VEX U chunks at Setup. Have them re-run with `CHUNKS = ALL_CHUNKS` and ask "how long is the autonomous period?" to watch a VEX U answer pollute a V5RC question, then put the filter back. Metadata filtering in one beat.